<a href="https://colab.research.google.com/github/annaphuongwit/14_LLMs/blob/main/1_LLM_KE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Building a Local Large Language Model Application

This guide provides instructions for setting up and running a local Large Language Model (LLM) application using Groq and LlamaIndex. You should already have a root folder for this project with `src` and `data` subdirectories, and a configured Miniconda environment with only Python installed.

> This notebook is a guide to setting up this project on your local machine. **Do not run the code cells here, you will get an error.**

---
## 1.&nbsp; Library Installation

First, we'll install the necessary Python libraries for our project. These commands should be executed in your VSCode terminal with your Conda environment activated.

In the VSCode terminal, check you have python installed

In [ ]:
#python --version

If this doesn't show you a version of python, install it.

In [ ]:
#conda install python

In the VSCode terminal, run the following commands to install `llama-index` and the specific [`groq`](https://docs.llamaindex.ai/en/stable/examples/llm/groq/) LLM library.

In [ ]:
#conda install conda-forge::llama-index

In [ ]:
#pip install llama-index-llms-groq

> You might wonder why llama-index-llms-groq is a separate installation. LlamaIndex has a modular design where the core library provides the main framework, and specific integrations for different LLMs (like Groq) are installed as separate packages. This approach keeps the framework lightweight by allowing you to install only the components you need.

Also in the VSCode terminal, run the following commands to install [`dotenv`](https://www.dotenv.org/docs/), a popular, zero-dependency module that loads environment variables from a `.env` file into `process.env`. This allows you to separate sensitive configuration, like API keys and database credentials, from your source code.

In [ ]:
#conda install -c conda-forge python-dotenv

> [conda-forge](https://conda-forge.org/) is a popular community-led channel that contains a vast collection of software packages. The `-c` flag is shorthand for --channel, telling Conda to look for the python-dotenv package in the conda-forge channel specifically. This often ensures you get the latest and most compatible version of the library.

---
## 2.&nbsp; API Key Configuration

To communicate with the Groq service, you need an API key. This key authenticates your requests to their servers. For security, we will store this key as an environment variable rather than hard-coding it into our scripts.

1. Navigate to the Groq Website
  * Open your web browser and go to the GroqCloud Console: [https://console.groq.com/](https://console.groq.com/)

2. Sign Up for a New Account
  * On the login page, you'll see options to "Continue with GitHub" or "Continue with Google".
  * Select your preferred method and follow the on-screen prompts to create and authorise your new account.

3. Access the API Keys Menu
  * Once you've successfully logged in, you'll be on the main dashboard.
  * Look for the menu on the left-hand side of the page. Click on `API Keys`.

4. Create a New API Key
  * On the API Keys page, click the `Create API Key` button, which is typically located in the top-right corner.
  * A form will appear. You can give your key a name to help you remember its purpose, though this is optional.
  * Click the `Create` button to generate your new key.

5. Copy Your New Key
  * A pop-up window will display your newly generated API key. This is the **only time** you will be able to see the full key.
  * Click the `Copy` button to copy the key to your clipboard. **Store it in a safe place immediately**, as you won't be able to view it again after closing this window.

We'll store our API key in a `.env` file. This is a secure, standard practice for managing secrets within a project.
1. Create the `.env` file
In your project's root folder (the same level as src and main.py), create a new file and name it exactly `.env`.

2. Add Your API Key
Open the new .env file and add the following line. Replace <your-api-key-here> with the key you copied from the Groq console.

In [ ]:
GROQ_API_KEY: str = ""

> **Important: Do not upload your `.env` file to GitHub.** This will give away all of your secrets!
>
> If you're using git, add `.env` to gitignore.

---
## 3.&nbsp; Project Structure and Configuration

We'll now create the necessary files and directories. This includes a configuration file to manage model parameters and Python package initialisers.

### 3.1 Create the Configuration File

1. In the VSCode file explorer, create a new file named `config.py` inside the `src` directory.
2. In the VSCode editor, add the following code to `src/config.py`. This file centralises all model parameters for easy management.
3. Save the file.

In [ ]:
# --- LLM Model Configuration ---
LLM_MODEL: str = "llama-3.1-8b-instant"
LLM_MAX_NEW_TOKENS: int = 768
LLM_TEMPERATURE: float = 0.01
LLM_TOP_P: float = 0.95
LLM_REPETITION_PENALTY: float = 1.03
LLM_QUESTION: str = "What is the capital of France?"

The `config.py` file serves as a centralised hub for all the settings and parameters that control your application's behaviour. Its primary purpose is to separate configuration from code.

Instead of hardcoding values like model names, file paths, or numerical settings directly into your functions, you define them as constants in `config.py`.

The key benefits of this approach are:

1. Easy Tweakability: You can easily change how the application works (e.g., switch to a different model) by editing just one file, without having to hunt through the logic in `engine.py`.

2. Improved Readability: It removes "magic numbers" and strings from your main code, making the logic cleaner and easier to understand.

3. Centralised Control: All the important variables that define your RAG pipeline's setup are in one predictable location, making the project much easier to manage and maintain.

### 3.2 Initialise the Python Package

In the VSCode file explorer, create an empty file named `__init__.py` inside the `src` directory.

While modern Python (3.3+) supports implicit namespace packages, including an `__init__.py` file is standard practice. It explicitly declares the directory as a package, which improves clarity, ensures robust imports, and enhances compatibility with development tools.

In [ ]:
# no need for setup.py to setup the src as a package?
# As long as you run Python from the project root (the directory above src), Python adds that root directory to sys.path.
# or Use PYTHONPATH to include src/

---
## 4.&nbsp; Create the Engine and its Components

Now we'll build the core of our application. Instead of putting all our logic in one file, we will apply a key software design principle called Separation of Concerns. This will make our project cleaner, more scalable, and easier to understand.

### 4.1 Separation of Concerns

Separation of Concerns means we should break down a complex application into distinct sections, where each section is responsible for one specific job or "concern".

In our project, we'll organise our code into two main files:

1. `components.py`

      * Concern: Initialisation.
      * Responsibility: Its only job is to create the fundamental, low-level tools our application needs. It knows how to connect to services by fetching API keys and which specific models to load. Think of it as our project's parts store; its functions produce the raw "parts".

2.  `engine.py`

      * Concern: Orchestration.
      * Responsibility: Its job is to take the "parts" created by `components.py` and assemble them into a final, working application. It's the assembly line where the individual components are put together to create the finished product: our chat engine.

By separating our code this way, we create a system that is more modular and easier to manage.

### 4.2 Create the Engine Directory Structure

First, let's create the necessary folder and files for our engine.

In the VSCode file explorer, perform the following steps:

1.  Inside the `src` directory, create a new folder named `engine`.
2.  Inside the new `src/engine` directory, create an empty `__init__.py` file.

Your structure should now look like this:

```
rag_project/
├── .env
├── data
├── main.py
└── src/
    ├── __init__.py
    ├── config.py
    └── engine/
        └── __init__.py
```

### 4.3 `components.py`

Let's create the file that will be responsible for initialising our components.

1.  In the VSCode file explorer, create a new file named `components.py` inside the `src/engine` directory.
2.  In the VSCode editor, add the following code to `src/engine/components.py`.
3.  Save the file.

In [ ]:
import os

print(os.getenv('my_name'))

Karim


In [ ]:
import os
# from dotenv import load_dotenv
from llama_index.llms.groq import Groq
# from src.config import (
#    LLM_MODEL,
    # LLM_MAX_NEW_TOKENS,
    # LLM_TEMPERATURE,
    # LLM_TOP_P,
    # LLM_REPETITION_PENALTY
# )

# Load environment variables from the .env file
# load_dotenv()

def initialise_llm() -> Groq:
    """Initialises the Groq LLM with core parameters from config."""

    api_key = os.getenv("GROQ_API_KEY")

    if not api_key:
        raise ValueError("GROQ_API_KEY not found. Make sure it's set in your .env file.")

    return Groq(
        api_key=api_key,
        model=LLM_MODEL,
        # The following parameters are optional and will default to the model's defaults if not set
        # max_new_tokens=LLM_MAX_NEW_TOKENS,
        # temperature=LLM_TEMPERATURE,
        # top_p=LLM_TOP_P,
        # repetition_penalty=LLM_REPETITION_PENALTY,
    )

/Users/karimelbana/.pyenv/versions/3.10.6/envs/rag_dsai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


> This script's only job, at the moment, is to create a configured `Groq` LLM object. It reads the API key from the environment and the model name from our `config.py` file, creating a reusable "component" for our application.

### 4.4 `engine.py`

Now we'll create the engine file that uses the component we just built.

1.  In the VSCode file explorer, create a new file named `engine.py` inside the `src/engine` directory.
2.  In the VSCode editor, add the following code to `src/engine/engine.py`.
3.  Save the file.

In [ ]:
from llama_index.core.llms import CompletionResponse
from llama_index.llms.groq import Groq
# from src.config import LLM_QUESTION
# from src.engine.components import initialise_llm

def main_chat_loop() -> None:
    """Main chat loop to ask a question to the LLM and print the answer."""

    llm: Groq = initialise_llm()

    api_key = os.getenv("GROQ_API_KEY")

    # answer: CompletionResponse = llm.complete(LLM_QUESTION)

    # Start the conversation loop
    while True:
        user_input = input("You 😎 (note: saying 'end' ends the chat ‼️): ")

        # Check for exit condition
        if user_input.lower() == "end": # USER= 'eNd'
        # if user_input.upper() == "END": USER= 'eNd'

            print("Ending the conversation. Goodbye!")
            break

        # Get the response from the conversation chain
        response = llm.complete(user_input)

        # Print the chatbot's response
        # print(response)

        # print(f"Question: {LLM_QUESTION}")
        print("-------------------")
        print(f"Answer 💁: {response}")

# there is no memory here

In [ ]:
main_chat_loop()

-------------------
Answer 💁: It seems like you forgot to ask a question. How can I assist you today?
-------------------
Answer 💁: It's nice to meet you. Is there something I can help you with or would you like to chat?
-------------------
Answer 💁: The Earth's diameter is approximately 12,742 kilometers (7,918 miles).
-------------------
Answer 💁: You didn't ask anything before this. This conversation just started. What would you like to talk about?
Ending the conversation. Goodbye!


In [ ]:
# Example 1: passing system prompt directly to Groq

import os

llm = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = llm.chat.completions.create(
    model="llama-3.3-70b-versatile",  # or whatever model you're using
    messages=[
        {ß
            "role": "system",
            "content": "You are a helpful assistant that explains things as simply as possible."
        },
        {
            "role": "user",
            "content": "How does photosynthesis work?"
        }
    ],
    temperature=0.7,
    top_p=0.9,
)

print(response.choices[0].message.content)

In [ ]:
# Example 2: passing a system prompt using llama_index SimpleChatEngine:

# Making a chatbot 💬
# To transform a basic LLM into a chatbot, we'll need to infuse it with additional functionalities: prompts, memory, and engines.

# **Prompts** are like the instructions you give the chatbot to tell it what to do. Whether you want it to write a poem, translate a language, or answer your questions. They provide the context and purpose for its responses.

# **Memory** is like the chatbot's brain. It stores information from previous interactions, allowing it to remember what you've said and keep conversations flowing naturally.

# The **engine** drives the other pieces of the chatbot. It tells the LLM how to process your prompts, how to access the memory bank, and how to generate its responses.

# In essence, prompts provide the direction, memory retains the context, and engines orchestrate the interactions.

from llama_index.core.chat_engine import SimpleChatEngine
from llama_index.core.base.llms.types import ChatMessage, MessageRole

llm: Groq = initialise_llm()

prompts = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="You are a nice chatbot having a conversation with a human.",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM, content="Keep your answers short and succinct."
    ),
]

conversation = SimpleChatEngine.from_defaults(llm=llm, prefix_messages=prompts)

# Here we see the explicit call of prompts and an engine, but where's the memory? Buried within the method `from_defaults()`, `ChatMemoryBuffer` is instantiated.

repsonse = conversation.chat("Tell me a joke.")
print(repsonse)

A man walked into a library and asked the librarian, "Do you have any books on Pavlov's dogs and Schrödinger's cat?" The librarian replied, "It rings a bell, but I'm not sure if it's here or not."


> Notice how clean this file is. It doesn't need to know about API keys or model names. Its only job is to orchestrate the application flow: it imports the `initialise_llm` function from our `components` module, calls it to get the LLM, and then uses the LLM to run the chat loop.

You've now successfully created the core engine logic with a clean separation of concerns. In the next step, we will create the main `main.py` file to run our application.

---
## 5.&nbsp; Create the Main Application Entry Point

A `main.py` file serves as the primary entry point for the application. This standard practice makes the project's starting point clear and easy to execute.

1. In the VSCode file explorer, create a new file named `main.py` in the project's root folder.
2. In the VSCode editor, add the following code to `main.py`.
3. Save the file.

In [ ]:
# from src.engine.engine import main_chat_loop

def main() -> None:
    print("--- 🤖 Main Application Starting ---")
    # Start the main chat session
    main_chat_loop()

if __name__ == "__main__":
    main()

--- 🤖 Main Application Starting ---
Question: What is the capital of France?
---
Answer: The capital of France is Paris.


> `if __name__ == "__main__"` is a standard and important convention in Python. It checks if the script is being executed directly by the user (e.g., python main.py).
>
> If it is, the code inside the block (in this case, calling our main() function) will run. This setup prevents the code from running automatically if this file is ever imported as a module into another script, making your code more reusable and predictable.

---
## 6.&nbsp; Run the Application

With all the components in place, run the application from the VSCode terminal.

In [ ]:
# python main.py

Congratulations, you've successfully built and run your LLM.

---
## 7.&nbsp; Challenges and Further Exploration 😀

Congratulations on building a functioning LLM application! The best way to learn is by experimenting. This section provides some challenges to help you explore the capabilities of your new application and understand the impact of different configurations.

Remember to **only change one parameter at a time**. This will help you isolate the effect of each change and develop a better intuition for how they influence the model's output.

### 1. Experiment with Different Models

Groq offers access to a variety of powerful open-source models, each with different strengths. You can significantly change the tone, style, and quality of the response by simply switching the model.

How to Change the Model:

1.  Open the `src/config.py` file in your VSCode editor.
2.  Locate the `LLM_MODEL` variable.
3.  Replace the current model ID with a new one from the list below.
4.  Make sure you save the changes to `src/config.py` or you won't change the model.

Where to Find Available Models:

You can find a complete and up-to-date list of available models on the [Groq Models Documentation Page](https://console.groq.com/docs/models).


**Challenge:** Try the same question (`LLM_QUESTION`)with at least three different models - maybe something trickier than just the capital of France will help you spot differences. Note the differences in speed, detail, and phrasing in their answers.

### 2. Adjust Model Parameters

The parameters in your `src/config.py` file give you fine-grained control over the model's response generation.

* `LLM_MAX_NEW_TOKENS`
    * What it does: This sets the absolute maximum number of tokens (which are like pieces of words) that the model is allowed to generate for its answer.
    * Experiment: Try setting this to a very low value (e.g., `50`) and see how the model cuts its response short. Then, try a much larger value (e.g., `2000`) and ask a complex question to see if it produces a longer, more detailed response.

* `LLM_TEMPERATURE`
    * What it does: This controls the randomness of the output. A higher temperature (e.g., `0.8`) makes the output more creative and unpredictable, as the model is more likely to choose less probable words. A lower temperature (e.g., `0.2`) makes the responses more deterministic and focused.
    * Experiment: Set the temperature to `0.9` and run the same prompt a few times. You should see more variation in the answers. Now, set it to `0.0` and run it again; the answers should be nearly identical each time.

* `LLM_TOP_P`
    * What it does: This is an alternative method to control randomness, often used instead of temperature. It works by selecting from the most probable words whose cumulative probability exceeds a certain threshold (`top_p`). A higher value (e.g., `0.95`) gives the model more words to choose from, leading to more creativity. A lower value (e.g., `0.5`) restricts the choices to a smaller, more probable set of words.
    * Experiment: Try setting `LLM_TEMPERATURE` to `1.0` and `LLM_TOP_P` to `0.1`. The model will be highly creative but will be forced to pick from a very narrow set of top words, which can produce interesting results.

* `LLM_REPETITION_PENALTY`
    * What it does: This parameter discourages the model from repeating the same words or phrases. A value greater than `1.0` (e.g., `1.2`) will apply a penalty to words that have already appeared, making them less likely to be chosen again.
    * Experiment: Ask the model a question that might lead to repetitive text. First, run it with the penalty at `1.0` (no penalty). Then, increase it to `1.3` and observe if the output becomes less repetitive and more diverse.

By methodically adjusting these settings, you will gain a practical understanding of how to tailor an LLM's behaviour to fit your specific needs. Happy experimenting!


![Screenshot 2025-09-16 at 23.25.53.png](<attachment:Screenshot 2025-09-16 at 23.25.53.png>)